# Kafka Producers

![Alt Text](/home/student/A2/FIT3182_A3/A3/34900403_33524815_assignment03/visuals/kafka_workflow_diagram.png)

First, import the the kafka3, pandas and time libraries. The KafkaProducer class and the sleep function would come from the kafak3 and time libraries respectively. From pymongo specifically you would need to import the MongoClient class. Import the json module as well and configure the host IP.

In [1]:
# Import necessary modules, classes and functions
from kafka import KafkaProducer
import pandas as pd
from time import sleep
import json

# Configure Host IP
hostip = "kafka"

The below function establishes a connection to the Kafka broker. This instantiates a KafkaProducer using the configured host IP address and port (9092).

In [2]:
def connect_kafka_producer():
    _producer = None
    try:
        _producer = KafkaProducer(bootstrap_servers=[f'{hostip}:29092'],
                                  api_version=(0, 10))
    except Exception as ex:
        print('Exception while connecting Kafka.')
        print(str(ex))
    
    return _producer

The below function establishes a connection to the Kafka broker. The message key and value payloads are casted to raw bytes using UTF-8 encoding, then pushed to the specified topic and finally, the function clears the network buffers.

In [3]:
def publish_message(producer_instance, topic_name, key, value):
    try:
        key_bytes = bytes(key, encoding='utf-8')
        value_bytes = bytes(value, encoding='utf-8')
        producer_instance.send(topic_name, key=key_bytes, value=value_bytes)
        producer_instance.flush()
        print('Message published successfully. Data: ' + str(value))
    except Exception as ex:
        print('Exception in publishing message.')
        print(str(ex))

Determine names for each of the three topics, one for each camera.

In [4]:
topic_a = 'camera_event_a'
topic_b = 'camera_event_b'
topic_c = 'camera_event_c'

Establish three independent producers, again, one for each camera.

In [5]:
producer_a = connect_kafka_producer()
producer_b = connect_kafka_producer()
producer_c = connect_kafka_producer()

Convert the CSV files containing the data for camera events for three different cameras into pandas dataframes.

In [6]:
# Obtain path directory locations for each CSV file

camera_event_a_path = "/home/jovyan/work/data/camera_event_A.csv"
camera_event_b_path = "/home/jovyan/work/data/camera_event_B.csv"
camera_event_c_path = "/home/jovyan/work/data/camera_event_C.csv"

# Convert the CSV files' content into pandas dataframes
camera_data_a = pd.read_csv(camera_event_a_path)
camera_data_b = pd.read_csv(camera_event_b_path)
camera_data_c = pd.read_csv(camera_event_c_path)

Initialize pointer trackers, localized storage arrays and state flag variables to synchronize/desync multi-stream batches across the three camera datasets.

In [7]:
# Track the current target batch ID sequence
curr_batch_id = 1

# Pointers start at 0
pointer_a = 0
pointer_b = 0
pointer_c = 0

# Arrays to compile row dictionaries for the active batch
batch_data_a = []
batch_data_b = []
batch_data_c = []

# Readiness flags start at False
batch_a_ok= False
batch_b_ok = False
batch_c_ok = False

The streaming loop scans through all the datasets in chronological order in a synchronized pointer-based manner. Rows are aggregate based on the active batch ID, stops the processing in the event of a boundary shift and converts the collected records into JSON before broadcasting them to their respective Kafka topics. Batches are also broadcast 1 second apart.

### Note
Only run this after running the last-most code cell from streaming_app.ipynb if you want the published messages to be added to the MongoDB collection.

In [8]:
def connect_kafka_producer():
    _producer = None
    try:
        _producer = KafkaProducer(
            bootstrap_servers=[f'{hostip}:29092'],  # was 9092, now 29092, used to resolve connection issues when running both containers
            api_version=(0, 10)
        )
    except Exception as ex:
        print('Exception while connecting Kafka.')
        print(str(ex))
    return _producer

def publish_message(producer_instance, topic_name, key, value):
    try:
        key_bytes = bytes(key, encoding='utf-8')
        value_bytes = bytes(value, encoding='utf-8')
        producer_instance.send(topic_name, key=key_bytes, value=value_bytes)
        producer_instance.flush()
        print('Message published successfully. Data: ' + str(value))
    except Exception as ex:
        print('Exception in publishing message.')
        print(str(ex))



topic_a = 'camera_event_a'
topic_b = 'camera_event_b'
topic_c = 'camera_event_c'

producer_a = connect_kafka_producer()
producer_b = connect_kafka_producer()
producer_c = connect_kafka_producer()

camera_event_a_path = "/home/jovyan/work/data/camera_event_A.csv"
camera_event_b_path = "/home/jovyan/work/data/camera_event_B.csv"
camera_event_c_path = "/home/jovyan/work/data/camera_event_C.csv"

camera_data_a = pd.read_csv(camera_event_a_path)
camera_data_b = pd.read_csv(camera_event_b_path)
camera_data_c = pd.read_csv(camera_event_c_path)

curr_batch_id = 1

pointer_a = 0
pointer_b = 0
pointer_c = 0

batch_data_a = []
batch_data_b = []
batch_data_c = []

batch_a_ok= False
batch_b_ok = False
batch_c_ok = False

while pointer_a < len(camera_data_a) or pointer_b < len(camera_data_b) or pointer_c < len(camera_data_c):

    # Gather batches of same batch ID across stream A
    if pointer_a < len(camera_data_a):
        # If a batch with different ID is found, set the batch_id to the new batch's ID
        if camera_data_a.iloc[pointer_a]["batch_id"] != curr_batch_id:
            batch_a_ok = True
        # If there are still records remaining in the current batch
        if not batch_a_ok:
            data = camera_data_a.iloc[pointer_a].to_dict()
            batch_data_a.append(data)
            pointer_a += 1
    
    # Gather batches of same batch ID across stream B
    if pointer_b < len(camera_data_b):
        if camera_data_b.iloc[pointer_b]["batch_id"] != curr_batch_id:
            batch_b_ok = True
        
        if not batch_b_ok:
            data = camera_data_b.iloc[pointer_b].to_dict()
            batch_data_b.append(data)
            pointer_b += 1

    # Gather batches of same batch ID across stream C
    if pointer_c < len(camera_data_c):
        if camera_data_c.iloc[pointer_c]["batch_id"] != curr_batch_id:
            batch_c_ok = True

        if not batch_c_ok:
            data = camera_data_c.iloc[pointer_c].to_dict()
            batch_data_c.append(data)
            pointer_c += 1
        
    # Once boundary flags reveal that all streams have finished gathering the current batch ID
    if batch_a_ok and batch_b_ok and batch_c_ok:

        # Convert the lists of records into JSON blocks
        batch_data_a = json.dumps(batch_data_a)
        batch_data_b = json.dumps(batch_data_b)
        batch_data_c = json.dumps(batch_data_c)

        # Obtain the Kafka key
        kafka_key = str(curr_batch_id)
        
        # Stream the compiled payloads out to their corresponding Kafka broker endpoints
        publish_message(producer_a, topic_a, kafka_key, batch_data_a)
        publish_message(producer_b, topic_b, kafka_key, batch_data_b)
        publish_message(producer_c, topic_c, kafka_key, batch_data_c)
        
        # Reset the batch arrays and readiness flags
        batch_data_a = []
        batch_data_b = []
        batch_data_c = []
        batch_a_ok= False
        batch_b_ok = False
        batch_c_ok = False

        # Move to the next batch index and pause for 1 second
        curr_batch_id += 1
        sleep(1)


Message published successfully. Data: [{"event_id": "d40c586c-5be6-4743-a1e3-2269d9edaa72", "batch_id": 1, "car_plate": "KRN 7", "camera_id": 1, "timestamp": "2024-01-01T08:00:04", "speed_reading": 77.2}, {"event_id": "85c08e3c-a0b5-45d8-a70c-df8f9a6d5829", "batch_id": 1, "car_plate": "ICE 8", "camera_id": 1, "timestamp": "2024-01-01T08:00:05", "speed_reading": 103.7}, {"event_id": "f5834b79-771b-4931-8da2-a5ad7f4ccd02", "batch_id": 1, "car_plate": "QE 1820", "camera_id": 1, "timestamp": "2024-01-01T08:00:03", "speed_reading": 67.4}, {"event_id": "d0e547bb-c4a7-4750-b7b4-8076e9b47f4f", "batch_id": 1, "car_plate": "CJW 924", "camera_id": 1, "timestamp": "2024-01-01T08:00:01", "speed_reading": 148.3}, {"event_id": "f3162606-1b2e-407f-951d-61d14c0a7b09", "batch_id": 1, "car_plate": "CJP 278", "camera_id": 1, "timestamp": "2024-01-01T08:00:02", "speed_reading": 125.2}, {"event_id": "c69852f7-cd9a-4892-b225-8f8a36ec017b", "batch_id": 1, "car_plate": "ZPG 90", "camera_id": 1, "timestamp": "2

Message published successfully. Data: [{"event_id": "d86e8cdb-c387-4ccb-b35a-346302238824", "batch_id": 1, "car_plate": "UTT 229", "camera_id": 3, "timestamp": "2024-01-01T08:00:54.958092", "speed_reading": 130.8}]


Message published successfully. Data: [{"event_id": "be770f2f-e15a-463c-b4a7-955ee3e14924", "batch_id": 2, "car_plate": "WB 418", "camera_id": 1, "timestamp": "2024-01-01T08:08:01", "speed_reading": 148.7}, {"event_id": "3930bb19-4837-4f01-a6b2-ce5757d1cc53", "batch_id": 2, "car_plate": "QJ 53", "camera_id": 1, "timestamp": "2024-01-01T08:08:02", "speed_reading": 82.6}, {"event_id": "4b1633e2-8268-4c82-a509-7dfde90635aa", "batch_id": 2, "car_plate": "EZ 6277", "camera_id": 1, "timestamp": "2024-01-01T08:08:02", "speed_reading": 93.5}, {"event_id": "d1fcb87d-b48c-43ea-9697-8e6ee3b83cf7", "batch_id": 2, "car_plate": "PI 9", "camera_id": 1, "timestamp": "2024-01-01T08:08:02", "speed_reading": 154.0}, {"event_id": "59e37658-17a9-41e6-8fd1-e65af3e271f9", "batch_id": 2, "car_plate": "VWM 13", "camera_id": 1, "timestamp": "2024-01-01T08:08:01", "speed_reading": 115.4}, {"event_id": "013b5963-adc3-426b-b6f4-2c649aef7630", "batch_id": 2, "car_plate": "VM 4837", "camera_id": 1, "timestamp": "202

Message published successfully. Data: [{"event_id": "b05ac078-6f38-41f4-a126-a7616870029d", "batch_id": 3, "car_plate": "CIY 810", "camera_id": 1, "timestamp": "2024-01-01T08:13:08", "speed_reading": 132.1}, {"event_id": "8d513b56-0bf4-467c-add2-77381a36f231", "batch_id": 3, "car_plate": "ARX 7573", "camera_id": 1, "timestamp": "2024-01-01T08:13:10", "speed_reading": 121.2}, {"event_id": "420c8cce-ca7d-48de-b99a-04dccf9d0b0b", "batch_id": 3, "car_plate": "GI 029", "camera_id": 1, "timestamp": "2024-01-01T08:13:10", "speed_reading": 60.8}, {"event_id": "379c53ab-ab19-448e-833b-c894e90b8ace", "batch_id": 3, "car_plate": "ZZ 8", "camera_id": 1, "timestamp": "2024-01-01T08:13:08", "speed_reading": 60.0}, {"event_id": "c14cd235-dd29-430d-8419-bd4e5e9c8b13", "batch_id": 3, "car_plate": "ZQ 22", "camera_id": 1, "timestamp": "2024-01-01T08:13:07", "speed_reading": 117.1}, {"event_id": "833f950f-1108-47ea-acde-f3de4ebf7af2", "batch_id": 3, "car_plate": "KWO 421", "camera_id": 1, "timestamp": "2

Message published successfully. Data: [{"event_id": "1dd988c0-d000-489b-8cf0-2cd738fb0565", "batch_id": 4, "car_plate": "BQN 88", "camera_id": 1, "timestamp": "2024-01-01T08:19:48", "speed_reading": 61.3}, {"event_id": "9478323b-a2f6-4592-8c9f-a9de5bb984ab", "batch_id": 4, "car_plate": "ZEA 3530", "camera_id": 1, "timestamp": "2024-01-01T08:19:48", "speed_reading": 153.3}, {"event_id": "3e613964-b717-44a8-bf81-a1290ae07a35", "batch_id": 4, "car_plate": "SJ 15", "camera_id": 1, "timestamp": "2024-01-01T08:19:48", "speed_reading": 146.2}, {"event_id": "9f538681-69f0-4efd-a3e3-ef44ee2c1094", "batch_id": 4, "car_plate": "PB 55", "camera_id": 1, "timestamp": "2024-01-01T08:19:49", "speed_reading": 111.2}, {"event_id": "26e1dd39-29ec-4692-bca0-5705579ed316", "batch_id": 4, "car_plate": "BU 9", "camera_id": 1, "timestamp": "2024-01-01T08:19:47", "speed_reading": 93.4}, {"event_id": "865113cf-ff8a-41ea-b9f6-48a76b8dc3c4", "batch_id": 4, "car_plate": "WX 49", "camera_id": 1, "timestamp": "2024-

...

Message published successfully. Data: [{"event_id": "379551c1-7c48-45ac-8349-3de844d6af96", "batch_id": 275, "car_plate": "BR 361", "camera_id": 1, "timestamp": "2024-01-03T19:47:00", "speed_reading": 117.8}, {"event_id": "a5ffcf66-7fa6-42c9-b3d4-95e061b9dbba", "batch_id": 275, "car_plate": "IC 7", "camera_id": 1, "timestamp": "2024-01-03T19:47:03", "speed_reading": 73.0}, {"event_id": "9e6cce21-07cf-41bb-b937-3f332b71f8fc", "batch_id": 275, "car_plate": "PWU 98", "camera_id": 1, "timestamp": "2024-01-03T19:47:04", "speed_reading": 111.2}, {"event_id": "73cd8e1d-e996-4d4d-83ff-2c3068ee56a9", "batch_id": 275, "car_plate": "IKB 3", "camera_id": 1, "timestamp": "2024-01-03T19:47:01", "speed_reading": 95.6}, {"event_id": "8b48a2e6-9596-49ee-948c-ab095ace076f", "batch_id": 275, "car_plate": "UI 36", "camera_id": 1, "timestamp": "2024-01-03T19:47:02", "speed_reading": 149.1}, {"event_id": "6cb2783b-3554-49dd-9aad-1d6093fababd", "batch_id": 275, "car_plate": "HY 5839", "camera_id": 1, "timest

Message published successfully. Data: [{"event_id": "6d6686f4-b048-44d5-b36f-8fab6375d3b1", "batch_id": 276, "car_plate": "YJ 1", "camera_id": 1, "timestamp": "2024-01-03T19:57:02", "speed_reading": 97.7}, {"event_id": "14d5eb36-d422-4b26-aa21-3a28377eb27f", "batch_id": 276, "car_plate": "QP 42", "camera_id": 1, "timestamp": "2024-01-03T19:57:01", "speed_reading": 76.8}, {"event_id": "5a8de845-32d3-42c5-bc0c-12925f14dbf8", "batch_id": 276, "car_plate": "QVX 9", "camera_id": 1, "timestamp": "2024-01-03T19:57:04", "speed_reading": 148.3}, {"event_id": "06b81d14-edf0-47b6-8ce9-f3ab8def8f5b", "batch_id": 276, "car_plate": "OPO 58", "camera_id": 1, "timestamp": "2024-01-03T19:57:01", "speed_reading": 79.0}, {"event_id": "7da50bee-640f-45b3-aa90-41af855a513e", "batch_id": 276, "car_plate": "QL 32", "camera_id": 1, "timestamp": "2024-01-03T19:57:01", "speed_reading": 77.6}, {"event_id": "6c4e6fdb-5493-41b1-9aa6-7f975df41705", "batch_id": 276, "car_plate": "XF 62", "camera_id": 1, "timestamp":

Message published successfully. Data: [{"event_id": "bc070a95-a93b-4423-a24d-2ac652d06c30", "batch_id": 277, "car_plate": "VPP 6", "camera_id": 1, "timestamp": "2024-01-03T20:04:48", "speed_reading": 84.2}, {"event_id": "29a1aad3-5625-4102-8f15-dc4437f81759", "batch_id": 277, "car_plate": "UJJ 9662", "camera_id": 1, "timestamp": "2024-01-03T20:04:44", "speed_reading": 61.5}, {"event_id": "f312a9e4-1501-484a-ab38-eb68b471fe20", "batch_id": 277, "car_plate": "QPI 3136", "camera_id": 1, "timestamp": "2024-01-03T20:04:46", "speed_reading": 123.2}, {"event_id": "42722872-99a8-431a-bf97-9c32d9cd3e58", "batch_id": 277, "car_plate": "OJ 083", "camera_id": 1, "timestamp": "2024-01-03T20:04:45", "speed_reading": 125.0}, {"event_id": "e106738a-8d68-40c6-b93e-746f4859dab4", "batch_id": 277, "car_plate": "CRU 95", "camera_id": 1, "timestamp": "2024-01-03T20:04:44", "speed_reading": 110.5}, {"event_id": "bbc91b17-8032-4a24-8365-7a2e57c7a316", "batch_id": 277, "car_plate": "FD 13", "camera_id": 1, "t

Message published successfully. Data: [{"event_id": "ae54502f-6aa6-443b-82dc-88696d7d7e82", "batch_id": 278, "car_plate": "JDS 9", "camera_id": 1, "timestamp": "2024-01-03T20:11:00", "speed_reading": 142.7}, {"event_id": "e21c9fbd-9399-4e85-b911-b8b469a6dc7f", "batch_id": 278, "car_plate": "YRA 4864", "camera_id": 1, "timestamp": "2024-01-03T20:11:05", "speed_reading": 99.2}, {"event_id": "104e4e36-0281-4c33-8d2e-da5883819fb4", "batch_id": 278, "car_plate": "VWI 0", "camera_id": 1, "timestamp": "2024-01-03T20:11:03", "speed_reading": 150.1}, {"event_id": "c03cf42d-d9c0-4843-bca2-9b42beaa97f4", "batch_id": 278, "car_plate": "TMZ 71", "camera_id": 1, "timestamp": "2024-01-03T20:11:05", "speed_reading": 136.1}, {"event_id": "279db505-20f7-41ce-934c-3193c83966d6", "batch_id": 278, "car_plate": "ZE 333", "camera_id": 1, "timestamp": "2024-01-03T20:11:05", "speed_reading": 130.5}, {"event_id": "a9c2cfb3-0f33-44cd-9b5b-67cf7abfb9f6", "batch_id": 278, "car_plate": "GJ 185", "camera_id": 1, "ti

Message published successfully. Data: [{"event_id": "b90dd3ba-3295-44a1-b729-1390f64c7c9b", "batch_id": 279, "car_plate": "SY 289", "camera_id": 1, "timestamp": "2024-01-03T20:17:46", "speed_reading": 77.7}, {"event_id": "f9be24e0-e86d-424b-9ec4-374e2726ff92", "batch_id": 279, "car_plate": "CPN 275", "camera_id": 1, "timestamp": "2024-01-03T20:17:47", "speed_reading": 89.5}, {"event_id": "4d449021-fc67-4e89-8a25-7578e4e650e9", "batch_id": 279, "car_plate": "CY 21", "camera_id": 1, "timestamp": "2024-01-03T20:17:47", "speed_reading": 75.3}, {"event_id": "c6afb30a-06b4-4dce-915b-fec1c63e31b5", "batch_id": 279, "car_plate": "WC 05", "camera_id": 1, "timestamp": "2024-01-03T20:17:48", "speed_reading": 153.5}, {"event_id": "af77f191-5a99-450e-84e5-91058c6b975b", "batch_id": 279, "car_plate": "ZI 517", "camera_id": 1, "timestamp": "2024-01-03T20:17:47", "speed_reading": 112.0}, {"event_id": "611aadf6-723f-4f40-b0d5-f1ada088e4f0", "batch_id": 279, "car_plate": "SKG 6", "camera_id": 1, "timest